In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": True, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.1f}'.format)


In [3]:
# trip data
trip = pd.read_csv(util.output_path / 'agg/dash/person_trips.csv')
# person data
person = pd.read_csv(util.output_path / 'agg/dash/person_geog.csv')
# vmt data
vmt = pd.read_csv(util.output_path / 'agg/dash/person_vmt.csv')

# list of equity geographies
equity_geogs = util.summary_config['hh_equity_geogs']
# not_equity_geogs = ["NOT in " + item for item in equity_geogs]

In [4]:
# TRIPS
df_trip = trip.copy()
# add home RGC
df_trip['is_rgc'] = 'Not in RGC'
df_trip.loc[df_trip['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
# add trip type
df_trip.loc[df_trip['dpurp'] != 'Work', 'trip_type'] = 'Non-Work'
df_trip.loc[df_trip['dpurp'] == 'Work', 'trip_type'] = 'Work'

# PERSONS
df_person = person.copy()
# add home RGC
df_person['is_rgc'] = 'Not in RGC'
df_person.loc[df_person['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

# VMT
df_vmt = vmt.copy()
# add home RGC
df_vmt['is_rgc'] = 'Not in RGC'
df_vmt.loc[df_vmt['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

# Select only walk and bike trips
df_vmt_bp = df_vmt[df_vmt['mode'].isin(['Walk','Bike'])].copy()
# Select only drivers (dorp = 1) and auto trips
df_vmt = df_vmt[df_vmt['mode'].isin(['SOV','HOV2','HOV3+']) & (df_vmt['dorp'] == 1)].copy()

# # not in equity geography
# df_person[not_equity_geogs] = 1 - df_person[equity_geogs]
# df_trip[not_equity_geogs] = 1 - df_trip[equity_geogs]
# df_vmt[not_equity_geogs] = 1 - df_vmt[equity_geogs]
# df_vmt_bp[not_equity_geogs] = 1 - df_vmt_bp[equity_geogs]

# total population by equity geography
equity_geogs_population = df_person[equity_geogs].apply(lambda x: x * df_person['psexpfac']).sum().reset_index()
equity_geogs_population.columns = ['Equity Group', 'psexpfac']

# total population by "NOT in" equity geography
# not_equity_geogs_population = df_person[not_equity_geogs].apply(lambda x: x * df_person['psexpfac']).sum().reset_index()
# not_equity_geogs_population.columns = ['Equity Group', 'psexpfac']

## Trips per Day by Resident

In [5]:
def trips_per_day(geog, map=False): 
    """
    Calculate trips per day by geography
    """

    # total population by geography
    df1 = df_person.groupby([geog], as_index=False)['psexpfac'].sum().set_index(geog)

    # total trips by geography
    df2 = df_trip.groupby([geog], as_index=False)['trexpfac'].sum().set_index(geog)

    # total trips by trip type and geography
    df3 = df_trip.groupby([geog, 'trip_type'], as_index=False)['trexpfac'].sum().set_index(geog)
    df3 = df3.pivot(columns='trip_type', values='trexpfac')

    # Merge the dataframes
    df = df1.merge(df2, left_index=True, right_index=True)
    df = df.merge(df3, left_index=True, right_index=True)

    if map:
        df.index = df.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })

    # regional totals
    df.loc['Region', ['Work','Non-Work','psexpfac','trexpfac']] = df[['Work','Non-Work','psexpfac','trexpfac']].sum()

    df['Total Trips per Day'] = df['trexpfac']/df['psexpfac']
    df['Work Trips per Day'] = df['Work']/df['psexpfac']
    df['Non-Work Trips per Day'] = df['Non-Work']/df['psexpfac']


    
    return df[['Work Trips per Day', 'Non-Work Trips per Day','Total Trips per Day']]

In [6]:
df = trips_per_day('hh_county')
df = df[df.index != 'Outside Region']
df

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_county,,,
King,0.5,3.3,3.8
Kitsap,0.4,3.2,3.7
Pierce,0.4,3.1,3.6
Snohomish,0.5,3.2,3.7
Region,0.5,3.2,3.7


In [7]:
trips_per_day('is_rgc')

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
is_rgc,,,
In RGC,0.7,3.3,4.0
Not in RGC,0.5,3.2,3.7
Region,0.5,3.2,3.7


In [8]:
trips_per_day('hh_rgc')

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_rgc,,,
Auburn,0.4,3.5,3.9
Bellevue,0.7,3.4,4.1
Bothell Canyon Park,0.6,3.3,3.9
Bremerton,0.5,3.5,4.0
Burien,0.5,3.3,3.8
Everett,0.5,3.4,4.0
Federal Way,0.4,3.5,3.9
Greater Downtown Kirkland,0.6,3.3,3.9
Kent,0.5,3.4,3.8


In [9]:
trips_per_day('hh_rg_proposed')

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_rg_proposed,,,
Cities and Towns,0.5,3.2,3.7
Core Cities,0.5,3.3,3.8
High Capacity Transit Communities,0.5,3.2,3.7
Metropolitan Cities,0.6,3.3,3.9
Rural Areas,0.4,3.1,3.5
Urban Unincorporated Areas,0.4,3.2,3.6
Region,0.5,3.2,3.7


### Equity Focus Areas

In [10]:
trips_per_day('hh_efa_poc', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_poc,,,
Below Regional Average,0.5,3.2,3.7
Above Regional Average,0.5,3.3,3.8
Higher Share of Equity Population,0.5,3.3,3.8
Region,0.5,3.2,3.7


In [11]:
trips_per_day('hh_efa_pov200', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_pov200,,,
Below Regional Average,0.5,3.2,3.8
Above Regional Average,0.5,3.2,3.7
Higher Share of Equity Population,0.5,3.2,3.7
Region,0.5,3.2,3.7


In [12]:
trips_per_day('hh_efa_lep', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_lep,,,
Below Regional Average,0.5,3.2,3.7
Above Regional Average,0.5,3.3,3.8
Higher Share of Equity Population,0.5,3.3,3.8
Region,0.5,3.2,3.7


In [13]:
trips_per_day('hh_efa_dis', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_dis,,,
Below Regional Average,0.5,3.3,3.8
Above Regional Average,0.5,3.2,3.7
Higher Share of Equity Population,0.4,3.2,3.7
Region,0.5,3.2,3.7


In [14]:
trips_per_day('hh_efa_older', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_older,,,
Below Regional Average,0.5,3.2,3.8
Above Regional Average,0.5,3.2,3.7
Higher Share of Equity Population,0.4,3.2,3.7
Region,0.5,3.2,3.7


In [15]:
trips_per_day('hh_efa_youth', map=True)

,Work Trips per Day,Non-Work Trips per Day,Total Trips per Day
hh_efa_youth,,,
Below Regional Average,0.5,3.3,3.8
Above Regional Average,0.5,3.2,3.7
Higher Share of Equity Population,0.4,3.2,3.6
Region,0.5,3.2,3.7


## Miles Driven per Day by Resident

In [16]:
# pd.options.display.float_format = '{:0,.1f}'.format
df_vmt = pd.read_csv(util.output_path / 'agg/dash/person_vmt.csv')
df_person = person.copy()

df_vmt['is_rgc'] = 'Not in RGC'
df_vmt.loc[df_vmt['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
df_person['is_rgc'] = 'Not in RGC'
df_person.loc[df_person['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'


# Select only drivers (dorp = 1) and auto trips
df_vmt = df_vmt[df_vmt['mode'].isin(['SOV','HOV2','HOV3+']) & (df_vmt['dorp'] == 1)]

def vmt_per_person(df_vmt, df_person, geog):
    _df_vmt = df_vmt.groupby(geog).sum()[['travdist_wt']]
    _df_person = df_person.groupby(geog).sum()[['psexpfac']]

    df = _df_vmt.merge(_df_person, left_index=True, right_index=True)
    df.loc['Region',:] = df.sum(axis=0)
    df['Average Miles per Person'] = df['travdist_wt']/df['psexpfac']
    
    return df[['Average Miles per Person']]

In [17]:
def miles_per_day(geog, map=False): 
    """
    Calculate trips per day by geography
    """

    # total population by geography
    df1 = df_person.groupby([geog], as_index=False)['psexpfac'].sum().set_index(geog)

    # total miles by geography
    df2 = df_vmt.groupby([geog], as_index=False)['travdist_wt'].sum().set_index(geog)

    # Merge the dataframes
    df = df1.merge(df2, left_index=True, right_index=True)

    if map:
        df.index = df.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })

    # regional totals
    df.loc['Region', ['psexpfac','travdist_wt']] = df[['psexpfac','travdist_wt']].sum()

    df['Average Miles per Person'] = df['travdist_wt']/df['psexpfac']
    
    return df[['Average Miles per Person']]

In [18]:
df = miles_per_day('hh_county')
df = df[df.index != 'Outside Region']
df

,Average Miles per Person
hh_county,
King,13.4
Kitsap,14.0
Pierce,15.1
Snohomish,16.4
Region,14.4


In [19]:
miles_per_day('is_rgc')

,Average Miles per Person
is_rgc,
In RGC,6.3
Not in RGC,14.9
Region,14.4


In [20]:
miles_per_day('hh_rgc')

,Average Miles per Person
hh_rgc,
Auburn,11.2
Bellevue,7.2
Bothell Canyon Park,15.7
Bremerton,7.1
Burien,12.4
Everett,8.6
Federal Way,11.3
Greater Downtown Kirkland,11.7
Kent,9.7


In [21]:
miles_per_day('hh_rg_proposed')

,Average Miles per Person
hh_rg_proposed,
Cities and Towns,17.5
Core Cities,13.9
High Capacity Transit Communities,14.8
Metropolitan Cities,10.0
Rural Areas,22.1
Urban Unincorporated Areas,16.0
Region,14.4


In [22]:
df = pd.DataFrame()
for name, col in {
    "People of Color": "hh_efa_poc",
    "Income": "hh_efa_pov200",
    "LEP": "hh_efa_lep",
    "Disability": "hh_efa_dis",
    "Older Adults": "hh_efa_older",
    "Youth": "hh_efa_youth"
}.items():
    df[name] = miles_per_day(col, map=True)
df

,People of Color,Income,LEP,Disability,Older Adults,Youth
hh_efa_poc,,,,,,
Below Regional Average,15.6,15.3,14.8,14.6,13.7,12.9
Above Regional Average,13.2,13.7,14.2,14.5,15.1,15.7
Higher Share of Equity Population,12.7,11.7,13.1,13.3,15.5,16.2
Region,14.4,14.4,14.4,14.4,14.4,14.4


## Delay

In [23]:
pd.options.display.float_format = '{:0,.1f}'.format

def delay_per_person(geog):

    df_person = pd.read_csv(util.output_path / 'agg/dash/person_geog.csv')
    df_person['is_rgc'] = 'Not in RGC'
    df_person.loc[df_person['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

    df = pd.read_csv(util.output_path / 'agg/dash/trip_time_total.csv')
    df['is_rgc'] = 'Not in RGC'
    df.loc[df['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
    df = df[(df['mode'].isin(['SOV','HOV2','HOV3+'])&(df['dorp']==1))]
    df = df.groupby(geog).sum()[['travtime_wt']]

    df2 = pd.read_csv(util.output_path / 'agg/dash/trip_sov_ff_time.csv')
    df2['is_rgc'] = 'Not in RGC'
    df2.loc[df2['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
    df2 = df2[(df2['mode'].isin(['SOV','HOV2','HOV3+'])&(df2['dorp']==1))]
    df2 = df2.groupby(geog).sum()[['sov_ff_time_wt']]
    df = df2.merge(df, on=geog)

    # Hours of delay from travel time (in min)
    df['Total Delay Hours'] = (df['travtime_wt'] - df['sov_ff_time_wt'])/60
    # Set any negative delay to 0
    df.loc[df['Total Delay Hours'] < 0, 'Total Delay Hours'] = 0

    df_person = df_person.groupby(geog).sum()[['psexpfac']]

    df = df.merge(df_person, left_index=True, right_index=True)
    df.loc['Region',:] = df.sum(axis=0)
    df['Average Minutes of Delay per Person'] = df['Total Delay Hours']/df['psexpfac']*60

    df['Annual Hours of Delay per Person'] = df['Average Minutes of Delay per Person']*util.summary_config['weekday_to_annual']/60

    df[['Total Delay Hours',
        'Annual Hours of Delay per Person']] = df[['Total Delay Hours', 'Annual Hours of Delay per Person']].astype(int).map('{:,}'.format)


    return df[['Total Delay Hours','Average Minutes of Delay per Person','Annual Hours of Delay per Person']]

df = delay_per_person('hh_county')
df = df[df.index != 'Outside Region']
df

,Total Delay Hours,Average Minutes of Delay per Person,Annual Hours of Delay per Person
hh_county,,,
King,"102,104",2.7,14
Kitsap,"5,457",1.2,6
Pierce,"35,642",2.3,12
Snohomish,"45,662",3.2,17
Region,"188,867",2.6,13


In [24]:
delay_per_person('is_rgc')

,Total Delay Hours,Average Minutes of Delay per Person,Annual Hours of Delay per Person
is_rgc,,,
In RGC,"5,140",1.1,5
Not in RGC,"183,726",2.7,14
Region,"188,867",2.6,13


In [25]:
df = delay_per_person('hh_rgc')

df

,Total Delay Hours,Average Minutes of Delay per Person,Annual Hours of Delay per Person
hh_rgc,,,
Auburn,56,1.5,8
Bellevue,454,1.6,8
Bothell Canyon Park,60,5.4,29
Bremerton,65,1.2,6
Burien,123,1.9,10
Everett,123,1.1,5
Federal Way,13,1.3,6
Greater Downtown Kirkland,448,3.3,17
Kent,63,1.8,9


In [26]:
delay_per_person('hh_rg_proposed')

,Total Delay Hours,Average Minutes of Delay per Person,Annual Hours of Delay per Person
hh_rg_proposed,,,
Cities and Towns,"18,070",2.8,14
Core Cities,"47,121",2.8,15
High Capacity Transit Communities,"50,239",3.1,16
Metropolitan Cities,"40,513",1.9,10
Rural Areas,"24,238",2.6,14
Urban Unincorporated Areas,"8,684",2.9,15
Region,"188,867",2.6,13
